# 4.01 注意力机制与掩码

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/04_transformer/4.01_attention.ipynb)


## 为什么要学

注意力（attention）是 Transformer 的核心。在 VLA 里：

- 动作 token 通过注意力"查看"图像 token 和文字 token，决定该怎么动；
- 注意力**掩码**决定了"谁能看谁"。π0 的设计精髓就是一个精心设计的**块状因果掩码**：图像和文字互相能看、但看不到动作；动作能看到所有东西。

本课从零实现注意力，并**逐行复现 openpi 的 `make_att_2d_masks`**。参考：[d2l 第 10 章 注意力机制](https://zh.d2l.ai/chapter_attention-mechanisms/index.html)。

---

## 1. 注意力 = 软性的"查字典"

普通字典：给一个 key，精确返回对应的 value。注意力：给一个**查询（query）** $\mathbf q$，与所有**键（key）** $\mathbf k_j$ 比较相似度，按相似度对所有**值（value）** $\mathbf v_j$ 加权平均：

$$
\text{Attention}(\mathbf q, K, V)=\sum_j\alpha_j\mathbf v_j,\qquad\alpha_j=\frac{\exp(\mathbf q^\top\mathbf k_j/\sqrt d)}{\sum_{j'}\exp(\mathbf q^\top\mathbf k_{j'}/\sqrt d)}
$$

写成矩阵形式（$n$ 个 query 一起算）：

$$
\boxed{\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt d}\right)V}\qquad Q\in\mathbb{R}^{n\times d},\ K\in\mathbb{R}^{m\times d},\ V\in\mathbb{R}^{m\times d_v}
$$

- $QK^\top$：$(n\times m)$ 的相似度矩阵，第 $i$ 行是第 $i$ 个 query 对所有 key 的打分；
- 除以 $\sqrt d$：1.05 课验证过，防止点积方差随维度增大、softmax 饱和；
- softmax 按**行**做：每个 query 的权重和为 1。

**自注意力**：$Q,K,V$ 都来自同一个序列 $X$（$Q=XW_Q$，$K=XW_K$，$V=XW_V$），每个 token 去看序列中所有 token。
**交叉注意力**：$Q$ 来自一个序列（如动作 token），$K,V$ 来自另一个序列（如图像特征）。ACT 的解码器、GR00T 的 DiT 动作头用的是交叉注意力；π0 则把所有 token 拼起来做自注意力，用掩码控制信息流向。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

### 练习 1：缩放点积注意力（带掩码）

掩码约定（与 openpi 一致）：`mask` 是布尔张量，`True` 表示**允许**关注。被禁止的位置在 softmax 之前填一个很大的负数，softmax 后权重变成 0。

In [ ]:
def attention(q, k, v, mask=None):
    """
    q: (..., n, d)  k: (..., m, d)  v: (..., m, dv)
    mask: 可广播到 (..., n, m) 的 bool 张量，True = 可以看
    返回 (输出 (..., n, dv), 注意力权重 (..., n, m))
    """
    # TODO:
    # 1) scores = q @ k^T / sqrt(d)
    # 2) 若有 mask：scores = scores.masked_fill(~mask, 一个很大的负数，例如 torch.finfo(scores.dtype).min)
    # 3) weights = softmax(scores, dim=-1)；out = weights @ v
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def attention(q, k, v, mask=None):
    """
    q: (..., n, d)  k: (..., m, d)  v: (..., m, dv)
    mask: 可广播到 (..., n, m) 的 bool 张量，True = 可以看
    返回 (输出 (..., n, dv), 注意力权重 (..., n, m))
    """
    d = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

In [ ]:
# ✅ 自测：与 PyTorch 官方实现对比
torch.manual_seed(0)
q, k, v = torch.randn(2, 4, 5, 16), torch.randn(2, 4, 7, 16), torch.randn(2, 4, 7, 8)
mask = torch.rand(5, 7) > 0.3; mask[:, 0] = True             # 保证每行至少能看一个位置
out, w = attention(q, k, v, mask)
ref = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
assert torch.allclose(out, ref, atol=1e-5)
assert torch.allclose(w.sum(-1), torch.ones(2, 4, 5)) and (w[..., ~mask] == 0).all()
print("通过 ✅")

### 直观感受：注意力在"找"什么

构造一个"查询找最相似的键"的例子：

In [ ]:
torch.manual_seed(1)
keys = torch.randn(6, 8)
query = keys[3:4] * 3 + 0.1 * torch.randn(1, 8)        # 和第 3 个 key 方向几乎一样
_, w = attention(query, keys, torch.eye(6))
plt.figure(figsize=(5, 1.8)); plt.bar(range(6), w[0].numpy()); plt.title("attention weights: query is similar to key #3"); plt.xlabel("key index"); plt.show()

---

## 2. 三种掩码

### (1) Padding 掩码

一个 batch 里句子长短不一，短的要补 `<pad>`。padding 位置**不应被任何 token 看到**。若 `pad_mask (B, L)` 中 True 表示真实 token，则二维掩码为

$$
M_{ij} = \text{pad}_i\wedge\text{pad}_j
$$

### (2) 因果掩码（Causal）

语言模型生成第 $i$ 个词时，只能看前面的词，不能"偷看答案"：$M_{ij}=[j\le i]$，即一个下三角矩阵。

### (3) 前缀语言模型掩码（Prefix-LM）

PaliGemma 的做法：前缀（图像 + 指令）内部**双向**注意力（互相都能看），后缀（要生成的回答）**因果**注意力，后缀能看全部前缀。

### 统一的写法：openpi / big_vision 的 `make_att_2d_masks`

openpi 用一个整数数组 `att_masks (B, N)` 统一描述以上所有情况：

> 1 表示"**从这里开始一个新的块，前面的 token 不能看我**"；0 表示"和前一个 token 同属一个块"。token $j$ 能被 token $i$ 看到，当且仅当 $\text{cumsum}(\text{att})_j\le\text{cumsum}(\text{att})_i$。

例子（来自 openpi 源码注释）：

- `[1 1 1 1 1 1]`：纯因果注意力；
- `[0 0 0 1 1 1]`：前缀语言模型，前 3 个双向，后 3 个因果；
- `[1 0 1 0 1 0 0 1 0 0]`：4 个块之间因果，块内双向。

### 练习 2：复现 `make_att_2d_masks`

In [ ]:
def make_att_2d_masks(pad_masks, att_masks):
    """
    pad_masks: bool (B, N)，True = 真实 token
    att_masks: int/bool (B, N)，1 = 新块开始
    返回 bool (B, N, N)：[b, i, j] = token i 能否看 token j
    """
    # TODO:
    # cumsum = torch.cumsum(att_masks, dim=1)                     (B, N)
    # att_2d = cumsum[:, None, :] <= cumsum[:, :, None]            (B, N, N)：j 的块号 <= i 的块号
    # pad_2d = pad_masks[:, None, :] & pad_masks[:, :, None]
    # return att_2d & pad_2d
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def make_att_2d_masks(pad_masks, att_masks):
    """
    pad_masks: bool (B, N)，True = 真实 token
    att_masks: int/bool (B, N)，1 = 新块开始
    返回 bool (B, N, N)：[b, i, j] = token i 能否看 token j
    """
    cumsum = torch.cumsum(att_masks.long(), dim=1)
    att_2d = cumsum[:, None, :] <= cumsum[:, :, None]
    pad_2d = pad_masks[:, None, :] & pad_masks[:, :, None]
    return att_2d & pad_2d

In [ ]:
# ✅ 自测：三个经典例子
ones = lambda n: torch.ones(1, n, dtype=torch.bool)
causal = make_att_2d_masks(ones(6), torch.tensor([[1, 1, 1, 1, 1, 1]]))[0]
assert torch.equal(causal, torch.tril(torch.ones(6, 6, dtype=torch.bool)))
prefix = make_att_2d_masks(ones(6), torch.tensor([[0, 0, 0, 1, 1, 1]]))[0]
assert prefix[:3, :3].all() and not prefix[:3, 3:].any() and prefix[3:, :3].all()
pad = make_att_2d_masks(torch.tensor([[True, True, False]]), torch.tensor([[0, 0, 0]]))[0]
assert not pad[:, 2].any() and not pad[2].any()
print("通过 ✅")

### π0 的注意力掩码长什么样？

π0 的 token 序列是：`[图像 token × 3 个相机] [文字 token] [状态 token] [动作 token × H]`。对照 `pi0_pytorch.py`：

- `embed_prefix`：图像和文字的 `att_masks` 全是 0 → 前缀内部完全双向；
- `embed_suffix`：状态 token 是 `[1]`（新块：前缀看不到它），动作 token 是 `[1, 0, 0, ...]`（又一个新块：前缀和状态看不到动作；动作之间双向）。
- π0.5 没有单独的状态 token（状态被离散化后放进文字前缀里），所以后缀只有动作块。

In [ ]:
def pi0_masks(n_img, n_txt, n_txt_valid, horizon, pi05=False):
    """按 openpi 的规则构造 pad_masks 与 att_masks（batch=1）"""
    pad = [True] * n_img + [True] * n_txt_valid + [False] * (n_txt - n_txt_valid)
    att = [0] * n_img + [0] * n_txt
    if not pi05:
        pad += [True]; att += [1]                  # 状态 token：新的一块
    pad += [True] * horizon; att += [1] + [0] * (horizon - 1)   # 动作块
    return torch.tensor([pad]), torch.tensor([att])

pad, att = pi0_masks(n_img=6, n_txt=5, n_txt_valid=3, horizon=4)
M = make_att_2d_masks(pad, att)[0]
labels = ["img"] * 6 + ["txt"] * 3 + ["pad"] * 2 + ["state"] + ["act"] * 4
plt.figure(figsize=(6, 6)); plt.imshow(M, cmap="Blues")
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7); plt.yticks(range(len(labels)), labels, fontsize=7)
plt.xlabel("key (who is looked at)"); plt.ylabel("query (who is looking)"); plt.title("pi0 attention mask (blue = allowed)"); plt.show()

读图：

- 左上（图像+文字）是一整块蓝色：**双向注意力**，这正是 VLM 预训练时的样子，所以能复用 PaliGemma 的权重；
- 图像/文字那几行的右边全白：**前缀看不到状态和动作**。这一点至关重要：前缀的表示与动作无关，**推理时前缀只需计算一次并缓存（KV cache）**，去噪的 10 步只重复计算动作部分（4.03 课实现 KV cache，7.02 课精读 openpi 的实现）；
- 动作行全蓝：动作能看到一切，包括其它动作（动作块内双向，不是因果的）。

### 练习 3：为什么前缀不能看动作？

用 `attention` 验证"前缀与后缀无关"这一性质：构造随机的 token 序列，改变后缀（动作 token）的内容，检查前缀 token 的注意力输出**完全不变**。

In [ ]:
def prefix_independent_of_suffix(mask, n_prefix, d=16, seed=0):
    """返回 True 表示：在该掩码下，改变后缀不会影响前缀的注意力输出"""
    torch.manual_seed(seed)
    N = mask.shape[-1]
    x = torch.randn(N, d)
    # TODO: 计算 out1 = attention(x, x, x, mask)[0]；把 x 的后缀部分换成新的随机数得到 x2，再算 out2；
    #       比较 out1[:n_prefix] 与 out2[:n_prefix] 是否相同
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def prefix_independent_of_suffix(mask, n_prefix, d=16, seed=0):
    """返回 True 表示：在该掩码下，改变后缀不会影响前缀的注意力输出"""
    torch.manual_seed(seed)
    N = mask.shape[-1]
    x = torch.randn(N, d)
    out1 = attention(x, x, x, mask)[0]
    x2 = x.clone(); x2[n_prefix:] = torch.randn(N - n_prefix, d)
    out2 = attention(x2, x2, x2, mask)[0]
    return torch.allclose(out1[:n_prefix], out2[:n_prefix], atol=1e-6)

In [ ]:
# ✅ 自测（这里用一个没有 padding 的例子：6 个图像 token + 5 个文字 token）
pad2, att2 = pi0_masks(n_img=6, n_txt=5, n_txt_valid=5, horizon=4)
M2 = make_att_2d_masks(pad2, att2)[0]
n_prefix = 11
assert prefix_independent_of_suffix(M2, n_prefix)
full = torch.ones_like(M2)                                    # 如果用全连接掩码……
assert not prefix_independent_of_suffix(full, n_prefix)
print("通过 ✅ —— 块状因果掩码保证了前缀可以只算一次")

> **为什么不直接用上面带 padding 的掩码 `M` 测试？** 因为 padding token 那几行**全是 False**（它什么都不能看）。整行被屏蔽时，softmax 会退化成对所有位置的均匀分布，于是 padding 位置的输出会"偷看"到后缀，自测就失败了。这不影响正确性：padding 位置的输出本来就没有意义，openpi 从来不使用它们（计算损失时只取动作 token 的输出）。但如果你自己写代码时**用 padding 位置的输出做了什么**，就会出现这种隐蔽的信息泄漏。

---

## 小结

- 注意力 = softmax($QK^\top/\sqrt d$)$V$：用相似度对 value 加权平均。
- 掩码控制信息流：padding、因果、前缀 LM，都可以用 openpi 的 `make_att_2d_masks(pad, att)` 统一表示（`att=1` 开启一个新块）。
- π0 的掩码：`[图像+文字]` 双向 → `[状态]` → `[动作块]`，前缀看不到后缀，所以前缀可以缓存。

**思考题**：如果把动作块的 `att_masks` 改成 `[1, 1, 1, ...]`（动作之间因果），模型会有什么变化？有什么 VLA 是这样做的？（提示：自回归生成离散动作 token 的 OpenVLA、π0-FAST。）